# 07 — Final Operational Evaluation & Report

**Project**: UPI Fraud Detection — Drift-Aware Explainable ML Framework
**Goal**: Consolidate all metrics, generate final evaluation report, compare all components

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import joblib
import json

from data_prep import load_ieee_cis, load_paysim, prepare_features_target, handle_class_imbalance
from models import evaluate_model, train_all_baselines
from evaluation import (
    compute_operational_metrics,
    compute_retraining_metrics,
    compare_strategies_statistically,
    generate_evaluation_report,
    find_optimal_threshold,
    compute_latency_metrics,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_ROOT = Path('../data')
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports')

print(f"Data root: {DATA_ROOT.absolute()}")

## 1. Load All Saved Metrics

In [ ]:
# Load baseline test metrics
baseline_files = {
    'ieee_test': REPORTS_DIR / 'ieee_test_metrics.csv',
    'paysim_test': REPORTS_DIR / 'paysim_test_metrics.csv',
    'ieee_cv': REPORTS_DIR / 'ieee_cv_summary.csv',
    'paysim_cv': REPORTS_DIR / 'paysim_cv_summary.csv',
}

baseline_metrics = {}
for name, path in baseline_files.items():
    if path.exists():
        baseline_metrics[name] = pd.read_csv(path)
        print(f"Loaded {name}: {baseline_metrics[name].shape}")
    else:
        print(f"Missing: {name} at {path}")

In [ ]:
# Load drift & SHAP results
drift_files = {
    'drift_alerts': REPORTS_DIR / 'drift_alerts_log.csv',
    'shap_importance': REPORTS_DIR / 'shap_importance_history.csv',
    'shap_stability': REPORTS_DIR / 'shap_stability.csv',
    'shap_global': REPORTS_DIR / 'shap_global_importance.csv',
}

drift_metrics = {}
for name, path in drift_files.items():
    if path.exists():
        drift_metrics[name] = pd.read_csv(path)
        print(f"Loaded {name}: {drift_metrics[name].shape}")
    else:
        print(f"Missing: {name} at {path}")

In [ ]:
# Load retraining results
retrain_files = {
    'strategy_comparison': REPORTS_DIR / 'retraining_strategy_comparison.csv',
    'detailed_results': REPORTS_DIR / 'retraining_detailed_results.csv',
}

retrain_metrics = {}
for name, path in retrain_files.items():
    if path.exists():
        retrain_metrics[name] = pd.read_csv(path)
        print(f"Loaded {name}: {retrain_metrics[name].shape}")
    else:
        print(f"Missing: {name} at {path}")

In [ ]:
# Load delayed labels results
delay_files = {
    'delayed_eval': REPORTS_DIR / 'delayed_labels_evaluation.csv',
    'delay_impact': REPORTS_DIR / 'delay_impact_summary.csv',
}

delay_metrics = {}
for name, path in delay_files.items():
    if path.exists():
        delay_metrics[name] = pd.read_csv(path)
        print(f"Loaded {name}: {delay_metrics[name].shape}")
    else:
        print(f"Missing: {name} at {path}")

## 2. Baseline Model Performance Summary

In [ ]:
if 'ieee_test' in baseline_metrics:
    print("=== IEEE-CIS Test Set Performance ===")
    display(baseline_metrics['ieee_test'][['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])  # type: ignore[unknown-name]
    
if 'paysim_test' in baseline_metrics:
    print("=== PaySim Test Set Performance ===")
    display(baseline_metrics['paysim_test'][['pr_auc', 'roc_auc', 'precision_at_100', 'recall_at_100']])  # type: ignore[unknown-name]


## 3. Cross-Validation Summary

In [ ]:
if 'ieee_cv' in baseline_metrics:
    print("=== IEEE-CIS Rolling CV Summary ===")
    display(baseline_metrics['ieee_cv'])  # type: ignore[unknown-name]

if 'paysim_cv' in baseline_metrics:
    print("=== PaySim Rolling CV Summary ===")
    display(baseline_metrics['paysim_cv'])  # type: ignore[unknown-name]


## 4. Delayed Labels Impact

In [ ]:
if 'delay_impact' in delay_metrics:
    print("=== Delay Impact Summary ===")
    display(delay_metrics['delay_impact'])  # type: ignore[unknown-name]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    df = delay_metrics['delay_impact']
    ax.plot(df['delay_days'], df['avg_pr_auc'], 'bo-', label='PR-AUC')
    ax.plot(df['delay_days'], df['avg_p100'], 'ro-', label='P@100')
    ax.set_xlabel('Label Delay (Days)')
    ax.set_ylabel('Metric')
    ax.set_title('Performance Degradation with Delayed Labels')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

## 5. Drift Monitoring Summary

In [ ]:
if 'drift_alerts' in drift_metrics:
    alerts = drift_metrics['drift_alerts']
    print(f"Total drift alerts: {len(alerts)}")
    if len(alerts) > 0:
        print("\nAlert type breakdown:")
        print(alerts['type'].value_counts())
        
        print("\nMost alerted features:")
        if 'column' in alerts.columns:
            print(alerts['column'].value_counts().head(10))
        
        print("\nAlert timeline:")
        alerts['timestamp_parsed'] = pd.to_datetime(alerts['timestamp'])
        daily_alerts = alerts.groupby(alerts['timestamp_parsed'].dt.date).size()
        print(daily_alerts)

## 6. SHAP Stability Summary

In [ ]:
if 'shap_stability' in drift_metrics:
    stability = drift_metrics['shap_stability']
    print("=== SHAP Stability (Top 20) ===")
    display(stability.head(20))  # type: ignore[unknown-name]
    
    stable = stability[stability['cv'] < 0.3]
    unstable = stability[stability['cv'] > 0.5]
    print(f"Stable features (CV < 0.3): {len(stable)}")
    print(f"Unstable features (CV > 0.5): {len(unstable)}")

## 7. Retraining Strategy Comparison

In [ ]:
if 'strategy_comparison' in retrain_metrics:
    print("=== Retraining Strategy Comparison ===")
    display(retrain_metrics['strategy_comparison'])  # type: ignore[unknown-name]

if 'detailed_results' in retrain_metrics:
    detailed = retrain_metrics['detailed_results']
    print(f"\nDetailed results: {len(detailed)} records")
    if not detailed.empty:
        stat_comp = compare_strategies_statistically(detailed, metric='pr_auc')
        print("\nStatistical Comparisons (PR-AUC):")
        for comp, result in stat_comp.items():
            sig = "SIGNIFICANT" if result['significant'] else "not significant"
            print(f"  {comp}: diff={result['mean_diff']:.4f}, p={result['p_value']:.4f} ({sig})")

## 8. Operational Metrics Dashboard

In [ ]:
print("="*60)
print("OPERATIONAL DASHBOARD SUMMARY")
print("="*60)

if 'ieee_test' in baseline_metrics:
    best_ieee = baseline_metrics['ieee_test'].loc[baseline_metrics['ieee_test']['pr_auc'].idxmax()]
    print(f"\nBest IEEE-CIS Model: {best_ieee.name}")
    print(f"  PR-AUC: {best_ieee['pr_auc']:.4f}")
    print(f"  P@100: {best_ieee.get('precision_at_100', 0):.4f}")
    print(f"  R@100: {best_ieee.get('recall_at_100', 0):.4f}")
    
    fp_100k = best_ieee.get('fp_per_100k_at_budget_100', 0)
    print(f"  FP/100k (budget=100): {fp_100k:.2f}")
    
    print(f"\n  TARGETS:")
    print(f"    PR-AUC > 0.85: {'PASS' if best_ieee['pr_auc'] > 0.85 else 'FAIL'} ({best_ieee['pr_auc']:.4f})")
    print(f"    P@100 > 0.70: {'PASS' if best_ieee.get('precision_at_100', 0) > 0.70 else 'FAIL'} ({best_ieee.get('precision_at_100', 0):.4f})")
    print(f"    FP/100k < 50: {'PASS' if fp_100k < 50 else 'FAIL'} ({fp_100k:.2f})")

## 9. Generate Final Report

In [ ]:
report_lines = [
    "="*70,
    "UPI FRAUD DETECTION - FINAL EVALUATION REPORT",
    "="*70,
    "",
    f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
    f"Student: Pavan C (USN: 4VV25MC060)",
    f"College: Vidyavardhaka College of Engineering",
    f"Course: MCA, Sem 3",
    "",
    "="*70,
    "1. PROJECT OVERVIEW",
    "="*70,
    "",
    "A drift-aware, explainable ML framework for fraud detection in UPI
digital payments under delayed labels. Uses IEEE-CIS and PaySim as
time-ordered proxies for private UPI data.",
    "",
    "Pipeline: Time-ordered split -> Baseline models -> Delayed labels sim ->
Drift monitoring (PSI/KS) -> SHAP explanations -> Retraining strategies ->
Operational evaluation",
    "",
    "="*70,
    "2. BASELINE MODEL PERFORMANCE (Test Set)",
    "="*70,
    "",
]

if 'ieee_test' in baseline_metrics:
    report_lines.extend(["", "IEEE-CIS:", ""])
    for _, row in baseline_metrics['ieee_test'].iterrows():
        report_lines.append(f"  {row.name}: PR-AUC={row['pr_auc']:.4f}, P@100={row.get('precision_at_100', 0):.4f}, R@100={row.get('recall_at_100', 0):.4f}")

if 'paysim_test' in baseline_metrics:
    report_lines.extend(["", "PaySim:", ""])
    for _, row in baseline_metrics['paysim_test'].iterrows():
        report_lines.append(f"  {row.name}: PR-AUC={row['pr_auc']:.4f}, P@100={row.get('precision_at_100', 0):.4f}, R@100={row.get('recall_at_100', 0):.4f}")

report_lines.extend([
    "",
    "="*70,
    "3. DELAYED LABELS IMPACT",
    "="*70,
    "",
    "Simulated 1/7/30-day label delays with rolling window evaluation.",
    "Performance degradation measured vs oracle (0-day delay).",
    "",
    "="*70,
    "4. DRIFT MONITORING (PSI/KS)",
    "="*70,
    "",
    "Label-free drift detection on input features and model scores.",
    "Thresholds: PSI > 0.2, KS > 0.05.",
    "",
    "="*70,
    "5. SHAP EXPLAINABILITY",
    "="*70,
    "",
    "Per-alert TreeSHAP explanations + global importance tracking.",
    "Stability measured via coefficient of variation (CV) over time.",
    "",
    "="*70,
    "6. RETRAINING STRATEGY COMPARISON",
    "="*70,
    "",
    "Three strategies: Static, Scheduled (weekly), Drift-Triggered.",
    "Evaluated on rolling windows with operational metrics.",
    "",
    "="*70,
    "7. TARGET ACHIEVEMENT",
    "="*70,
    "",
    "Target: PR-AUC > 0.85, P@100 > 0.70, FP/100k < 50",
    "Retraining: Drift-triggered 30% fewer than scheduled",
    "SHAP latency: < 200ms/alert",
    "",
    "="*70,
])

report_text = "\n".join(report_lines)
print(report_text)

with open(REPORTS_DIR / 'final_evaluation_report.txt', 'w') as f:
    f.write(report_text)
print(f"\nReport saved to {REPORTS_DIR / 'final_evaluation_report.txt'}")

## 10. Summary & Project Completion

In [ ]:
print("""
=== PROJECT COMPLETE ===

✅ All 7 notebooks executed
✅ Baseline models trained & evaluated
✅ Delayed labels simulation completed
✅ Drift monitoring (PSI/KS) operational
✅ SHAP explanations generated
✅ Retraining strategies compared
✅ Final evaluation report generated

NEXT STEPS:
1. Run dashboard: streamlit run dashboard/app.py
2. Build Docker: docker build -t upi-fraud .
3. Push to GitHub with README
4. Update presentation PPTX with results
""")